# Your Title Here

**Name(s)**: (your name(s) here)

**Website Link**: (your website link)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [3]:
league2022 = pd.read_csv("league2022.csv")
league2022.head(10)



/var/folders/46/6br2dnxn5874qvdyzpchl4x00000gn/T/ipykernel_2187/1278933924.py:1: DtypeWarning:

Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.



,gameid,datacompleteness,url,league,...,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,1.0,0.0,2.0,0.0
1,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,1.0,5.0,1.0
2,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,0.0,3.0,4.0,3.0
...,...,...,...,...,...,...,...,...,...
7,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,3.0,1.0,3.0,0.0
8,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,0.0,2.0,1.0,2.0
9,ESPORTSTMNT01_2690210,complete,NaN,LCKC,...,2.0,1.0,3.0,2.0


## Step 2: Data Cleaning and Exploratory Data Analysis

In [ ]:

columns = [
    "gameid",
    "participantid",
    "side",
    "deathsat10",
    "position",
    "csdiffat10",
    "golddiffat10",
    "xpdiffat10",
    "csdiffat20",
    "golddiffat20",
    "xpdiffat20"
]

data = league2022[columns].copy()

print("Original dataset shape:", data.shape)

position_mapping = {
    "TOP": "Top",
    "JNG": "Jungle",
    "MID": "Mid",
    "BOT": "ADC",
    "ADC": "ADC",
    "SUP": "Support",
    "SUPPORT": "Support"
}

data["position"] = (
    data["position"]
    .astype("string")
    .str.strip()
    .str.upper()
    .map(position_mapping)
)

valid_positions = ["Top", "Jungle", "Mid", "ADC", "Support"]

data = data[data["position"].isin(valid_positions)].copy()

numeric_columns = [
    "deathsat10",
    "csdiffat10",
    "golddiffat10",
    "xpdiffat10",
    "csdiffat20",
    "golddiffat20",
    "xpdiffat20"
]

for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data = data.dropna(
    subset=[
        "deathsat10",
        "golddiffat20"
    ]
)

# Verify cleaned data
print("Cleaned dataset shape:", data.shape)

print("\nPosition counts:")
print(data["position"].value_counts())

print("\nUnique positions:")
print(data["position"].unique())

data.head()


Original dataset shape: (150228, 11)
Cleaned dataset shape: (105950, 11)

Position counts:
position
Top        21190
Jungle     21190
Mid        21190
ADC        21190
Support    21190
Name: count, dtype: int64

Unique positions:
['Top' 'Jungle' 'Mid' 'ADC' 'Support']


np.float64(6.0)

## Step 3: Assessment of Missingness

In [12]:
# Calculate the percentage of players for each death count by position
death_percentages = pd.crosstab(
    data["position"],
    data["deathsat10"],
    normalize="index"
) * 100

# Keep positions in the correct order
death_percentages = death_percentages.reindex(
    ["Top", "Jungle", "Mid", "ADC", "Support"]
)

print(death_percentages.round(2))

# Reset index and convert to long format
plot_data = death_percentages.reset_index()

plot_data = plot_data.rename(columns={"position": "Position"})

plot_data = plot_data.melt(
    id_vars="Position",
    var_name="Deaths at 10 Minutes",
    value_name="Percentage"
)

# Create stacked bar chart
fig = px.bar(
    plot_data,
    x="Position",
    y="Percentage",
    color="Deaths at 10 Minutes",
    title="Percentage Distribution of Early Deaths by Position",
    category_orders={
        "Position": ["Top", "Jungle", "Mid", "ADC", "Support"],
        "Deaths at 10 Minutes": [0, 1, 2, 3, 4, 5, 6]
    },
    labels={
        "Position": "Player Position",
        "Percentage": "Percentage of Players (%)",
        "Deaths at 10 Minutes": "Early Deaths"
    },
    barmode="stack",
    color_discrete_sequence=px.colors.sequential.RdPu
)

fig.update_layout(
    yaxis_range=[0, 100],
    yaxis_title="Percentage of Players (%)",
    xaxis_title="Player Position",
    legend_title="Deaths at 10 Minutes"
)

fig.show()

deathsat10    0.0    1.0   2.0   3.0   4.0   5.0   6.0
position                                              
Top         64.69  26.89  7.00  1.24  0.16  0.01  0.01
Jungle      62.24  29.64  7.00  0.97  0.15  0.00  0.00
Mid         70.51  23.65  4.93  0.80  0.10  0.00  0.00
ADC         69.23  24.04  5.68  0.90  0.14  0.02  0.00
Support     61.13  29.16  7.99  1.45  0.25  0.01  0.00


## Step 4: Hypothesis Testing

In [13]:
# Create gold difference categories
data["Gold Status at 20"] = data["golddiffat20"].apply(
    lambda x: "Gold Disadvantage" if x < 0
    else "Even" if x == 0
    else "Gold Advantage"
)

# Calculate percentages by position
gold_percentages = pd.crosstab(
    data["position"],
    data["Gold Status at 20"],
    normalize="index"
) * 100

gold_percentages = gold_percentages.reindex(
    ["Top", "Jungle", "Mid", "ADC", "Support"]
)

print(gold_percentages.round(2))

# Convert to long format
plot_data = gold_percentages.reset_index().rename(
    columns={"position": "Position"}
)

plot_data = plot_data.melt(
    id_vars="Position",
    var_name="Gold Status at 20 Minutes",
    value_name="Percentage"
)

# Create chart
fig = px.bar(
    plot_data,
    x="Position",
    y="Percentage",
    color="Gold Status at 20 Minutes",
    title="Distribution of Gold Advantage at 20 Minutes by Position",
    category_orders={
        "Position": ["Top", "Jungle", "Mid", "ADC", "Support"],
        "Gold Status at 20 Minutes": [
            "Gold Disadvantage",
            "Even",
            "Gold Advantage"
        ]
    },
    labels={
        "Position": "Player Position",
        "Percentage": "Percentage of Players (%)",
        "Gold Status at 20 Minutes": "Gold Status"
    },
    barmode="stack",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_layout(
    yaxis_range=[0, 100],
    yaxis_title="Percentage of Players (%)",
    xaxis_title="Player Position",
    legend_title="Gold Status"
)

fig.show()

Gold Status at 20  Even  Gold Advantage  Gold Disadvantage
position                                                  
Top                0.00           50.00              50.00
Jungle             0.02           49.99              49.99
Mid                0.02           49.99              49.99
ADC                0.01           50.00              50.00
Support            0.08           49.96              49.96


In [19]:
import plotly.express as px

# Remove rows missing either variable
plot_data = data.dropna(
    subset=["deathsat10", "golddiffat20", "position"]
).copy()

# Make death counts categorical for easier comparison
plot_data["deathsat10"] = plot_data["deathsat10"].astype(int).astype(str)

# Order death counts numerically
death_order = sorted(
    plot_data["deathsat10"].unique(),
    key=int
)

# Create faceted box plots
fig = px.box(
    plot_data,
    x="deathsat10",
    y="golddiffat20",
    facet_col="position",
    facet_col_wrap=3,
    category_orders={
        "position": ["Top", "Jungle", "Mid", "ADC", "Support"],
        "deathsat10": death_order
    },
    points=False,
    title="Gold Difference at 20 Minutes by Early Death Count and Position",
    labels={
        "deathsat10": "Deaths by 10 Minutes",
        "golddiffat20": "Gold Difference at 20 Minutes",
        "position": "Player Position"
    }
)

# Add reference line for zero gold difference
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="red"
)

fig.update_layout(
    height=750,
    showlegend=False,
    title_x=0.5
)

fig.update_yaxes(title_text="Gold Difference at 20 Minutes")
fig.update_xaxes(title_text="Deaths by 10 Minutes")

fig.show()

## Step 5: Framing a Prediction Problem

In [7]:
# TODO

## Step 6: Baseline Model

In [8]:
# TODO

## Step 7: Final Model

In [9]:
# TODO

## Step 8: Fairness Analysis

In [10]:
# TODO